In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pickle
import tensorflow as tf
import keras
from keras import ops
import random

from sklearn.preprocessing import MinMaxScaler
from robot_vlp.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR, TRAINING_LOGS_DIR

import robot_vlp.data.preprocessing as p

%load_ext autoreload
%autoreload 2


2024-11-16 21:38:10.253 | INFO     | robot_vlp.config:<module>:11 - PROJ_ROOT path is: /Users/tyrelglass/PhD/Repositories/robot-vlp


In [2]:
import robot_vlp.modeling.train_rnn as rn


In [8]:
with open(PROCESSED_DATA_DIR/'odometer_navigated_data.pkl', 'rb') as handle:
    data = pickle.load(handle)

In [27]:
tuner = rn.build_random_search_tuner(TRAINING_LOGS_DIR, 'odo_nav_rnd_search', overwrite = True)

In [28]:
tensorboard_cb = tf.keras.callbacks.TensorBoard(TRAINING_LOGS_DIR / 'odo_nav_rnd_search/tensorboard')
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=5)

In [ ]:
tuner.search(x = data['X_valid'],
    y = [data['y_valid'][:,[0,1]],  p.ang_to_vector(data['y_valid'][:,2], unit = 'degrees').numpy()],
    epochs = 10,
    validation_data = (data['X_test'], [data['y_test'][:,[0,1]], p.ang_to_vector(data['y_test'][:,2], unit = 'degrees').numpy()]), 
    callbacks = [early_stopping_cb, tensorboard_cb]
    )


Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
4                 |4                 |n_hidden
10                |10                |n_neurons
0.00065625        |0.00065625        |learning_rate
adam              |adam              |optimizer
gru               |gru               |layer_type

Epoch 1/10
4257/4257 ━━━━━━━━━━━━━━━━━━━━ 98s 22ms/step - heading_loss: 0.7209 - loss: 2.3230 - pos_loss: 1.6020 - val_heading_loss: 0.1167 - val_loss: 0.1809 - val_pos_loss: 0.0642
Epoch 2/10
4257/4257 ━━━━━━━━━━━━━━━━━━━━ 97s 23ms/step - heading_loss: 0.1119 - loss: 0.1680 - pos_loss: 0.0561 - val_heading_loss: 0.0810 - val_loss: 0.1317 - val_pos_loss: 0.0507
Epoch 3/10
4257/4257 ━━━━━━━━━━━━━━━━━━━━ 102s 24ms/step - heading_loss: 0.0834 - loss: 0.1306 - pos_loss: 0.0472 - val_heading_loss: 0.0753 - val_loss: 0.1238 - val_pos_loss: 0.0484
Epoch 4/10
4257/4257 ━━━━━━━━━━━━━━━━━━━━ 100s 23ms/step - heading_loss: 0.0723 - loss: 0.1154 - pos_loss: 0.0431 - val_heading_

In [13]:
tuner.get_best_models()

[<Functional name=functional, built=True>]

In [17]:
tuner.get_best_hyperparameters(num_trials= 2)[0].values

{'n_hidden': 3,
 'n_neurons': 140,
 'learning_rate': 0.00013607174450468629,
 'optimizer': 'adam',
 'layer_type': 'simple'}

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=./my_logs